In [6]:
import kagglehub;
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from google.colab import files
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import seaborn as sns
import os;

In [7]:
# Download latest version
path = kagglehub.dataset_download("asaniczka/amazon-kindle-books-dataset-2023-130k-books")
# path = '/root/.cache/kagglehub/datasets/asaniczka/amazon-kindle-books-dataset-2023-130k-books/versions/35';
print("Path to dataset files:", path)

files = os.listdir(path) 
print("Files in this folder:", files)
full_file_path = os.path.join(path, files[0])

df = pd.read_csv(full_file_path)

Using Colab cache for faster access to the 'amazon-kindle-books-dataset-2023-130k-books' dataset.
Path to dataset files: /kaggle/input/amazon-kindle-books-dataset-2023-130k-books
Files in this folder: ['kindle_data-v2.csv']


In [8]:
# Display the shape of the dataset
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(133102, 16)


In [9]:
# Display concise summary of the DataFrame
print("Dataset Info:")
print(df.info())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133102 entries, 0 to 133101
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   asin               133102 non-null  object 
 1   title              133102 non-null  object 
 2   author             132677 non-null  object 
 3   soldBy             123869 non-null  object 
 4   imgUrl             133102 non-null  object 
 5   productURL         133102 non-null  object 
 6   stars              133102 non-null  float64
 7   reviews            133102 non-null  int64  
 8   price              133102 non-null  float64
 9   isKindleUnlimited  133102 non-null  bool   
 10  category_id        133102 non-null  int64  
 11  isBestSeller       133102 non-null  bool   
 12  isEditorsPick      133102 non-null  bool   
 13  isGoodReadsChoice  133102 non-null  bool   
 14  publishedDate      84086 non-null   object 
 15  category_name      133102 non-null  o

In [10]:
# Display the first few rows
print("First 5 Rows:")
display(df.head())

First 5 Rows:


,asin,title,author,soldBy,imgUrl,productURL,stars,reviews,price,isKindleUnlimited,category_id,isBestSeller,isEditorsPick,isGoodReadsChoice,publishedDate,category_name
0,B00TZE87S4,Adult Children of Emotionally Immature Parents...,Lindsay C. Gibson,Amazon.com Services LLC,https://m.media-amazon.com/images/I/713KZTsaYp...,https://www.amazon.com/dp/B00TZE87S4,4.8,0,9.99,False,6,True,False,False,2015-06-01,Parenting & Relationships
1,B08WCKY8MB,"From Strength to Strength: Finding Success, Ha...",Arthur C. Brooks,Penguin Group (USA) LLC,https://m.media-amazon.com/images/I/A1LZcJFs9E...,https://www.amazon.com/dp/B08WCKY8MB,4.4,0,16.99,False,6,False,False,False,2022-02-15,Parenting & Relationships
2,B09KPS84CJ,Good Inside: A Guide to Becoming the Parent Yo...,Becky Kennedy,HarperCollins Publishers,https://m.media-amazon.com/images/I/71RIWM0sv6...,https://www.amazon.com/dp/B09KPS84CJ,4.8,0,16.99,False,6,False,True,False,2022-09-13,Parenting & Relationships
3,B07S7QPG6J,Everything I Know About Love: A Memoir,Dolly Alderton,HarperCollins Publishers,https://m.media-amazon.com/images/I/71QdQpTiKZ...,https://www.amazon.com/dp/B07S7QPG6J,4.2,0,9.95,True,6,False,True,False,2020-02-25,Parenting & Relationships
4,B00N6PEQV0,The Seven Principles for Making Marriage Work:...,John Gottman,Random House LLC,https://m.media-amazon.com/images/I/813o4WOs+w...,https://www.amazon.com/dp/B00N6PEQV0,4.7,0,13.99,False,6,False,False,False,2015-05-05,Parenting & Relationships


In [11]:
	# Check for null values and display the count per column
print(df.isnull().sum())

asin                     0
title                    0
author                 425
soldBy                9233
imgUrl                   0
productURL               0
stars                    0
reviews                  0
price                    0
isKindleUnlimited        0
category_id              0
isBestSeller             0
isEditorsPick            0
isGoodReadsChoice        0
publishedDate        49016
category_name            0
dtype: int64


In [13]:
# Fill missing values
# For 'author' and 'soldBy', you could fill with a placeholder like 'Unknown'
df.fillna({'soldBy': 'Unknown'}, inplace=True)
df.fillna({'author': 'Unknown'}, inplace=True)

In [14]:
df.shape

(133102, 16)

In [15]:
# For 'publishedDate', convert to datetime and then drop rows with NaT (missing) values
df['publishedDate'] = pd.to_datetime(df['publishedDate'], errors='coerce');
df.dropna(subset=['publishedDate'], inplace=True);

In [16]:
print("nNull values count per column after handling:")
print(df.isnull().sum())
print("nDataset Info:")
print(df.info())

nNull values count per column after handling:
asin                 0
title                0
author               0
soldBy               0
imgUrl               0
productURL           0
stars                0
reviews              0
price                0
isKindleUnlimited    0
category_id          0
isBestSeller         0
isEditorsPick        0
isGoodReadsChoice    0
publishedDate        0
category_name        0
dtype: int64
nDataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 84086 entries, 0 to 133101
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   asin               84086 non-null  object        
 1   title              84086 non-null  object        
 2   author             84086 non-null  object        
 3   soldBy             84086 non-null  object        
 4   imgUrl             84086 non-null  object        
 5   productURL         84086 non-null  object        
 6   stars   